# SquigDecode: Signal-to-Base Decoder – Project Overview

This notebook provides a high-level overview of the **SquigDecode** project, a deep learning basecaller for high-SNR nanopore-style squiggle signals.

## Goals
- Introduce the project motivation and problem setting.
- Summarize the simulation, model, training, and validation pipeline.
- Show how configuration is controlled via `src/config.py` and `src/input.json`.

## Pipeline summary
- **Signal simulation** (`src/data_simulator.py`):
  - Uses physics-inspired parameters from `src/config.py` to generate synthetic nanopore squiggle signals and ground-truth DNA sequences.
  - Applies base-specific picoampere levels, variable dwell times, drift, noise, and smoothing.
  - Saves explicit **train** and **test** splits under `data/train` and `data/test`.

- **Quality control & visualization** (`src/QC_squig_data.py`):
  - Loads the generated signals and sequences.
  - Produces QC plots (signal traces with base annotations, dwell-time distributions, base composition, length statistics) saved in `results/`.

- **Model & training** (`src/architecture.py`, `src/train.py`):
  - Defines **SquigNet**, a CNN–BiLSTM hybrid model trained with **CTC Loss** for sequence-to-sequence basecalling.
  - Uses the `data/train` split with variable noise per sequence for robustness.
  - Saves checkpoints to `checkpoints/` and the final model weights to `models/squig_model.pt`, plus a loss curve plot.

- **Inference & validation** (`src/inference.py`):
  - Loads the trained model and validates it on the held-out `data/test` split.
  - Computes edit-distance-based accuracy and can optionally run an adversarial single-sample test with additional noise.

In [1]:
"""Inspect core configuration and user overrides for SquigDecode.

This cell makes `src` importable and prints out both the default
configuration (from `config.py`) and any overrides supplied via
`src/input.json` (exposed as `config.USER_CONFIG`).
"""
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

# Ensure src/ is on sys.path
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import config  # type: ignore

print("Project root:", PROJECT_ROOT)
print("Config file:", SRC_DIR / "config.py")
print("Input JSON:", SRC_DIR / "input.json")

print("\n=== Default simulation parameters (config) ===")
print("NUM_SEQUENCES:", config.NUM_SEQUENCES)
print("LENGTH RANGE:", config.MIN_LENGTH, "-", config.MAX_LENGTH)
print("DWELL_TIME_MEAN:", config.DWELL_TIME_MEAN)
print("DWELL_TIME_STD:", config.DWELL_TIME_STD)
print("NOISE_STD (baseline):", config.NOISE_STD)
print("DRIFT_FACTOR:", config.DRIFT_FACTOR)

print("\n=== Dataset split defaults (config) ===")
print("TRAIN_RATIO (default):", config.DATASET_TRAIN_RATIO)
print("RANDOM_SEED (default):", config.DATASET_RANDOM_SEED)
print(
    "TRAIN_NOISE_STD_RANGE (default):",
    config.DATASET_TRAIN_NOISE_STD_MIN,
    "..",
    config.DATASET_TRAIN_NOISE_STD_MAX,
)

print("\n=== User overrides from input.json (if any) ===")
user_cfg: dict[str, Any] = config.USER_CONFIG
print(json.dumps(user_cfg, indent=2))

Project root: /Users/janetx/Desktop/Janet/Projects/SquigDecode
Config file: /Users/janetx/Desktop/Janet/Projects/SquigDecode/src/config.py
Input JSON: /Users/janetx/Desktop/Janet/Projects/SquigDecode/src/input.json

=== Default simulation parameters (config) ===
NUM_SEQUENCES: 1000
LENGTH RANGE: 50 - 100
DWELL_TIME_MEAN: 15
DWELL_TIME_STD: 4
NOISE_STD (baseline): 3.5
DRIFT_FACTOR: 0.01

=== Dataset split defaults (config) ===
TRAIN_RATIO (default): 0.8
RANDOM_SEED (default): 42
TRAIN_NOISE_STD_RANGE (default): 3.5 .. 3.5

=== User overrides from input.json (if any) ===
{
  "num_sequences": 1500,
  "min_length": 50,
  "max_length": 100,
  "noise_std": 3.0,
  "adversarial_noise": 0.03,
  "drift_factor": 0.01,
  "train_ratio": 0.8,
  "random_seed": 42,
  "train_noise_std_min": 1.5,
  "train_noise_std_max": 3.0,
  "train_num_epochs": 50,
  "train_batch_size": 32,
  "train_learning_rate": 0.001,
  "checkpoint_dir": "checkpoints",
  "model_dir": "models",
  "data_dir": "data/train",
  "test_

## Dataset layout

After running:

```bash
python src/data_simulator.py
python src/train.py
```

the repository will contain:

- `data/train/signals.pt`, `data/train/sequences.pkl`, `data/train/dwell_times.pkl`, `data/train/metadata.pkl`
- `data/test/signals.pt`, `data/test/sequences.pkl`, `data/test/dwell_times.pkl`, `data/test/metadata.pkl`
- `models/squig_model.pt` (trained model weights)

These are consumed by:

- `src/train.py` (training on `data/train`)
- `src/QC_squig_data.py` (QC on `data/train` by default)
- `src/inference.py` (validation on `data/test` by default)